In [1]:
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import col

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MMDS") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/26 17:51:04 WARN Utils: Your hostname, MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.234 instead (on interface en0)
25/12/26 17:51:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/26 17:51:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
schema_ratings = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("item_id", IntegerType(), False),
    StructField("rating", IntegerType(), False),
    StructField("timestamp", IntegerType(), False)
])

schema_movies = StructType([
    StructField("item_id", IntegerType(), False),
    StructField("title", StringType(), False)
])

In [24]:
train_ratings = spark.read.option("delimiter", "::").csv("./data/ratings_train.dat", schema=schema_ratings)
test_ratings = spark.read.option("delimiter", "::").csv("./data/ratings_test.dat", schema=schema_ratings)
movies = spark.read.option("delimiter", "::").csv("./data/movies.dat", schema=schema_movies)

In [25]:
train_ratings.printSchema()
movies.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

root
 |-- item_id: integer (nullable = true)
 |-- title: string (nullable = true)



In [26]:
print(f"Number of training ratings: {train_ratings.count()}")
print(f"Number of test ratings: {test_ratings.count()}")
print(f"Number of movies: {movies.count()}")

Number of training ratings: 802553
Number of test ratings: 197656
Number of movies: 3883


In [27]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

als = ALS(maxIter=20, regParam=0.05, userCol="user_id", itemCol="item_id", ratingCol="rating",
          coldStartStrategy="drop", rank=15)

model = als.fit(train_ratings)

predictions = model.transform(test_ratings)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))


Root-mean-square error = 0.8786285982748049


In [28]:
users = test_ratings.select(als.getUserCol()).distinct().limit(3)
userSubsetRecs = model.recommendForUserSubset(users, 10)

In [29]:
userRecsFlat = (
    userSubsetRecs
    .withColumn("rec", explode("recommendations"))
    .select(
        col("user_id"),
        col("rec.item_id").alias("item_id"),
        col("rec.rating").alias("rating")
    )
)

In [40]:
userRecsWithTitles = (
    userRecsFlat
    .join(movies.select("item_id", "title"), on="item_id", how="left")
)

seen = train_ratings.select("user_id", "item_id")

userRecsWithTitles = (
    userRecsWithTitles
    .join(seen, ["user_id", "item_id"], "left_anti")
)

userRecsWithTitles \
    .orderBy("user_id", col("rating").desc()) \
    .show(50, truncate=False)

+-------+-------+---------+-------------------------------------------------------+
|user_id|item_id|rating   |title                                                  |
+-------+-------+---------+-------------------------------------------------------+
|68     |3851   |6.0141897|I'm the One That I Want (2000)                         |
|68     |1458   |5.775758 |Touch (1997)                                           |
|68     |3817   |5.7508893|Other Side of Sunday, The (S�ndagsengler) (1996)       |
|68     |718    |5.72843  |Visitors, The (Les Visiteurs) (1993)                   |
|68     |3949   |5.697846 |Requiem for a Dream (2000)                             |
|68     |3612   |5.5894027|Slipper and the Rose, The (1976)                       |
|68     |2813   |5.580679 |Source, The (1999)                                     |
|68     |2972   |5.531351 |Red Sorghum (Hong Gao Liang) (1987)                    |
|68     |116    |5.4893427|Anne Frank Remembered (1995)                     